# Classification metrics

Accuracy answers "how often is the model right?" That is rarely the question. A bank wants to know how many defaults it catches and how many good customers it turns away. A clinic wants to know how many cancers it misses. This notebook builds the metrics that answer those questions, shows why accuracy can be 96% for a model that does nothing, and ends with picking a threshold from the costs.

Order: intuition, maths, cost function, from-scratch version, library version, tuning, what breaks.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.calibration import calibration_curve
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from common.datasets import load_breast_cancer_data, make_credit_data

SEED = 42
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=3, suppress=True)
plt.rcParams["figure.figsize"] = (6, 3.5)

## 1. Intuition

A classifier makes two kinds of mistakes and they are rarely equally costly. Flag a good customer as a default and you lose a bit of margin. Miss a real default and you lose the loan. Treat "positive" as the case you are trying to catch (a default, a malignant tumour, a fraud).

Every metric here is a different way of weighing those two mistakes. And most classifiers do not output a label at all. They output a score, and a **threshold** turns the score into a label. Moving the threshold trades one mistake for the other. So there are two jobs: judge how good the score is across all thresholds (ROC, PR, AUC), and then choose the threshold that suits the costs.

## 2. Maths

Count the four outcomes of a labelled test set: true positives (TP), false positives (FP), true negatives (TN) and false negatives (FN). That table is the **confusion matrix**.

| Metric | Formula | Question it answers |
|---|---|---|
| Accuracy | $\frac{TP+TN}{n}$ | How often is the label right? |
| Precision | $\frac{TP}{TP+FP}$ | Of those flagged, how many were real? |
| Recall (TPR, sensitivity) | $\frac{TP}{TP+FN}$ | Of the real ones, how many did we catch? |
| Specificity | $\frac{TN}{TN+FP}$ | Of the real negatives, how many did we clear? |
| FPR | $\frac{FP}{FP+TN}=1-\text{specificity}$ | Of the real negatives, how many raised a false alarm? |
| F1 | $\frac{2PR}{P+R}$ | One number for precision and recall together |

F1 is the harmonic mean, which is dragged down by whichever of the two is lower, so you cannot hide a weak recall behind a strong precision. $F_\beta$ weights recall $\beta$ times as much as precision.

**Curves.** Lowering the threshold flags more cases, so recall rises and precision usually falls. The **ROC curve** plots TPR against FPR over all thresholds. **AUC** is the area under it, and has a neat reading: it is the probability that a random positive gets a higher score than a random negative. 0.5 is chance and 1.0 is perfect ranking. The **PR curve** plots precision against recall, and **average precision** is the area under it.

**Choosing a threshold.** If a false positive costs $c_{FP}$ and a false negative costs $c_{FN}$, flag a case when the expected cost of not flagging exceeds the cost of flagging: $p\,c_{FN}>(1-p)\,c_{FP}$. That gives the threshold

$$t^*=\frac{c_{FP}}{c_{FP}+c_{FN}}$$

which equals 0.5 only when both costs are equal. It assumes the probabilities are **calibrated**, meaning that among cases given $p=0.2$, about 20% are really positive.

## 3. Accuracy hides everything: the credit data

The synthetic credit data have about 4% defaults (see `common/datasets.py`). I fit a logistic regression with the default threshold and a "model" that always says no default.

In [ ]:
Xc, yc = make_credit_data(seed=SEED)
Xc = Xc.fillna(Xc.median())
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.3, random_state=SEED, stratify=yc)
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xc_tr, yc_tr)
p_te = lr.predict_proba(Xc_te)[:, 1]
pred = (p_te >= 0.5).astype(int)
lazy = np.zeros(len(yc_te), dtype=int)

for name, yhat in [("always no default", lazy), ("logistic at 0.5", pred)]:
    print(
        f"{name:>18}: accuracy {(yhat == yc_te).mean():.3f}  recall {recall_score(yc_te, yhat):.3f}  "
        f"precision {precision_score(yc_te, yhat, zero_division=0):.3f}"
    )
print("default rate in test set:", round(yc_te.mean(), 3))

Doing nothing scores 96% accuracy and catches no defaults at all. The real model barely moves accuracy and catches very few defaults at the 0.5 threshold. A base rate of 4% means a model has to be very confident before the score passes 0.5 for anyone, so almost nobody is flagged. Accuracy cannot tell the two apart, but recall can.

The confusion matrix shows where the mistakes go.

In [ ]:
cm = confusion_matrix(yc_te, pred)
tn, fp, fn, tp = cm.ravel()
print(f"TN {tn}  FP {fp}  FN {fn}  TP {tp}")
precision = tp / (tp + fp) if tp + fp else 0.0
recall = tp / (tp + fn)
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
print(f"by hand: precision {precision:.3f}, recall {recall:.3f}, F1 {f1:.3f}")
print(
    f"sklearn: precision {precision_score(yc_te, pred, zero_division=0):.3f}, recall {recall_score(yc_te, pred):.3f}, F1 {f1_score(yc_te, pred):.3f}"
)
ConfusionMatrixDisplay(cm, display_labels=["no default", "default"]).plot(colorbar=False)
plt.show()

## 4. ROC, AUC and the PR curve

These ignore the threshold and judge the ranking. First ROC and AUC from scratch: sort by score, then sweep the threshold from high to low, tracking TPR and FPR.

In [ ]:
def roc_scratch(y: np.ndarray, score: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    order = np.argsort(-score)
    y_sorted = y[order]
    tpr = np.r_[0, np.cumsum(y_sorted) / y.sum()]
    fpr = np.r_[0, np.cumsum(1 - y_sorted) / (1 - y).sum()]
    return fpr, tpr


y_arr = yc_te.to_numpy()
fpr, tpr = roc_scratch(y_arr, p_te)
auc_scratch = np.trapezoid(tpr, fpr)
# the rank reading of AUC: P(random positive scores above random negative)
pos, neg = p_te[y_arr == 1], p_te[y_arr == 0]
auc_rank = (pos[:, None] > neg[None, :]).mean() + 0.5 * (pos[:, None] == neg[None, :]).mean()
print(
    f"AUC scratch {auc_scratch:.4f}  rank form {auc_rank:.4f}  sklearn {roc_auc_score(y_arr, p_te):.4f}"
)

All three agree. An AUC of 0.76 says that a random defaulter outscores a random good customer 76% of the time. That is a modest but realistic ranking for credit data.

Now compare two models on both curves: the logistic regression and a gradient boosting model.

In [ ]:
gb = HistGradientBoostingClassifier(max_iter=100, random_state=SEED).fit(Xc_tr, yc_tr)
scores = {"logistic": p_te, "hist gradient boosting": gb.predict_proba(Xc_te)[:, 1]}
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name, s in scores.items():
    f, t, _ = roc_curve(y_arr, s)
    pr, rc, _ = precision_recall_curve(y_arr, s)
    axes[0].plot(f, t, label=f"{name} (AUC {roc_auc_score(y_arr, s):.3f})")
    axes[1].plot(rc, pr, label=f"{name} (AP {average_precision_score(y_arr, s):.3f})")
axes[0].plot([0, 1], [0, 1], "k--", lw=1)
axes[0].set(xlabel="false positive rate", ylabel="true positive rate", title="ROC")
axes[1].axhline(y_arr.mean(), c="k", ls="--", lw=1)
axes[1].set(xlabel="recall", ylabel="precision", title="Precision-recall (dashed: base rate)")
for ax in axes:
    ax.legend()
plt.show()

Compare the two views. The ROC curve bows clearly above the diagonal, because the FPR is measured against the huge pool of negatives, so even hundreds of false alarms barely move it to the right. The PR curve is harsher: precision falls quickly towards the 4% base rate, and the average precision is only 0.17. When positives are rare, **the PR curve is the more honest picture** of what you will experience, because precision cares about the false alarms in absolute terms. Logistic regression beats gradient boosting here (AUC 0.76 against 0.72, AP 0.17 against 0.11), because this data was generated from a logistic model and the boosted trees chase noise in 14,000 training rows with only about 540 defaults to learn from.

## 5. Choosing a threshold

The 0.5 default is arbitrary. Sweep the threshold and watch precision and recall trade.

In [ ]:
prec, rec, thr = precision_recall_curve(y_arr, p_te)
f1s = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
best = int(np.argmax(f1s))
print(
    f"F1 is highest at threshold {thr[best]:.3f}: precision {prec[best]:.3f}, recall {rec[best]:.3f}, F1 {f1s[best]:.3f}"
)
plt.plot(thr, prec[:-1], label="precision")
plt.plot(thr, rec[:-1], label="recall")
plt.plot(thr, f1s, label="F1")
plt.axvline(0.5, c="grey", ls=":")
plt.xlim(0, 0.5)
plt.xlabel("threshold")
plt.legend()
plt.show()

The F1-optimal threshold is about 0.14, far below 0.5. Even then precision and recall are both near 0.25, so this is a hard problem. Still, F1 treats the two mistakes as equally bad, and they are not. Use the costs.

Say a missed default costs \$8,000 (the loan lost) and a false alarm costs \$400 (a good customer's margin lost). Then $t^*=400/(400+8000)\approx0.048$. Compute the total cost on the test set across thresholds.

In [ ]:
c_fp, c_fn = 400, 8000
grid = np.linspace(0.005, 0.5, 100)
costs = []
for t in grid:
    flagged = p_te >= t
    costs.append(c_fp * np.sum(flagged & (y_arr == 0)) + c_fn * np.sum(~flagged & (y_arr == 1)))
t_best = grid[int(np.argmin(costs))]
print(f"theory t* = {c_fp / (c_fp + c_fn):.3f}; lowest-cost threshold on the grid = {t_best:.3f}")
print(
    f"cost at 0.5: ${costs[-1]:,.0f}   at best: ${min(costs):,.0f}   flag nobody: ${c_fn * y_arr.sum():,.0f}"
)
plt.plot(grid, costs)
plt.axvline(c_fp / (c_fp + c_fn), c="grey", ls=":")
plt.xlabel("threshold")
plt.ylabel("total cost ($)")
plt.show()

The best threshold from the grid is close to the theoretical $t^*$, and it cuts the cost from \$1.80 million at 0.5 (almost the same as flagging nobody, \$1.82 million) to \$1.24 million, about 31% lower. The dotted line shows $t^*$. The model did not change. Only the decision rule did, and that is where the money is. Choose the threshold on a validation set, never on the test set, and report the final cost on the test set.

## 6. Calibration

The $t^*$ rule only works if "0.2" means 20%. A reliability curve bins the predictions and compares the average prediction with the actual rate in each bin.

In [ ]:
bal = make_pipeline(
    StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced")
).fit(Xc_tr, yc_tr)
fig, ax = plt.subplots(figsize=(5, 4.5))
for name, s in [("logistic", p_te), ("class_weight=balanced", bal.predict_proba(Xc_te)[:, 1])]:
    frac, mean_p = calibration_curve(y_arr, s, n_bins=8, strategy="quantile")
    ax.plot(mean_p, frac, marker="o", label=f"{name} (Brier {brier_score_loss(y_arr, s):.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=1)
ax.set(xlabel="mean predicted probability", ylabel="observed default rate")
ax.legend()
plt.show()

The plain logistic model sits close to the diagonal, because the loss it minimises (log-loss) rewards honest probabilities. The reweighted one sits far above the diagonal: its middle bins claim 34% to 49% risk where the true default rate is 1% to 4%. It ranks equally well, so the AUC is unchanged, yet its probabilities are not probabilities any more. If you need a threshold from costs, or a price from the risk, do not reweight, or recalibrate afterwards with `CalibratedClassifierCV`. The Brier score (mean squared error of the probabilities) shows it: 0.034 for the plain model against 0.192 for the reweighted one.

## 7. What breaks

- **Accuracy on skewed classes.** Shown above. Report recall and precision, or the PR curve.
- **A fixed threshold.** The best threshold depends on the costs and on the base rate. If the population changes, it moves.
- **AUC on its own.** It says nothing about calibration or about the part of the curve you will operate in. Two models with equal AUC can differ a lot at a low FPR.
- **Many classes.** Precision and recall are computed per class. `average="macro"` weights every class equally and `"micro"` pools all the decisions, so macro is the one that shows a failing rare class.
- **Small test sets.** With 40 positives, one extra catch moves recall by 2.5 points. Report an interval, or use cross-validation.

A last look at a balanced, real dataset, where accuracy is not misleading. On breast cancer, 37% of cases are malignant, and recall on malignant is the number that matters.

In [ ]:
Xb, yb = load_breast_cancer_data()
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(Xb, yb, test_size=0.3, random_state=SEED, stratify=yb)
m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(Xb_tr, yb_tr)
print(
    classification_report(yb_te, m.predict(Xb_te), target_names=["benign", "malignant"], digits=3)
)

Even here, a missed malignant case (a false negative) is far worse than a false alarm, so a clinic would lower the threshold below 0.5 and accept more benign recalls to catch every malignant one. The next notebook puts all of this together in a credit scorecard.